# Play Obstacle Environments

Choose an environment with the buttons, preview an `rgb_array` frame inline, then run the last cell to play in a pygame window.

In [ ]:
import ipywidgets as widgets
import numpy as np
from IPython.display import display
from PIL import Image

VALID_ENV_NAMES = ("Obstacle", "ObstacleV2", "ObstacleV3", "ObstacleV4")

def make_env(env_name, **kwargs):
    if env_name == "Obstacle":
        from masa.envs.continuous.obstacle import Obstacle

        return Obstacle(**kwargs)
    if env_name == "ObstacleV2":
        from masa.envs.continuous.obstacle_v2 import ObstacleV2

        return ObstacleV2(**kwargs)
    if env_name == "ObstacleV3":
        from masa.envs.continuous.obstacle_v3 import ObstacleV3

        return ObstacleV3(**kwargs)
    if env_name == "ObstacleV4":
        from masa.envs.continuous.obstacle_v4 import ObstacleV4

        return ObstacleV4(**kwargs)
    raise ValueError(f"env_name must be one of {VALID_ENV_NAMES!r}")

ENV_SELECTOR = widgets.ToggleButtons(
    options=VALID_ENV_NAMES,
    value="ObstacleV2",
    description="Env",
)
display(ENV_SELECTOR)

ENV_NAME = ENV_SELECTOR.value
SEED = 0


In [ ]:
ENV_NAME = ENV_SELECTOR.value
preview_env = make_env(ENV_NAME, render_mode="rgb_array", render_window_size=512)
obs, info = preview_env.reset(seed=SEED)
print("reset:", {"obs": obs, "info": info})
display(Image.fromarray(preview_env.render()))
preview_env.close()


In [ ]:
def action_from_key(key, pygame):
    if key in (pygame.K_LEFT, pygame.K_a):
        return np.array([-1.0, 0.0], dtype=np.float32)
    if key in (pygame.K_RIGHT, pygame.K_d):
        return np.array([1.0, 0.0], dtype=np.float32)
    if key in (pygame.K_UP, pygame.K_w):
        return np.array([0.0, 1.0], dtype=np.float32)
    if key in (pygame.K_DOWN, pygame.K_s):
        return np.array([0.0, -1.0], dtype=np.float32)
    if key == pygame.K_SPACE:
        return np.array([0.0, 0.0], dtype=np.float32)
    return None

def play_env(env_name=None, seed=SEED):
    import pygame

    if env_name is None:
        env_name = ENV_SELECTOR.value
    env = make_env(env_name, render_mode="human", render_window_size=512)
    obs, info = env.reset(seed=seed)
    finished = False
    print("Controls: arrows or WASD accelerate, Space neutral, R resets, Q or Escape quits.")
    print("reset:", {"obs": obs, "info": info})

    try:
        running = True
        while running and not env.human_window_closed:
            action = None
            for event in pygame.event.get():
                if not env.handle_pygame_event(event):
                    running = False
                    break
                if event.type != pygame.KEYDOWN:
                    continue
                if event.key in (pygame.K_q, pygame.K_ESCAPE):
                    running = False
                    break
                if event.key == pygame.K_r:
                    obs, info = env.reset(seed=seed)
                    finished = False
                    print("reset:", {"obs": obs, "info": info})
                    continue
                if not finished:
                    action = action_from_key(event.key, pygame)

            if action is None:
                env.render()
                continue

            obs, reward, terminated, truncated, info = env.step(action)
            finished = terminated or truncated
            print({"obs": obs, "reward": reward, "terminated": terminated, "truncated": truncated, "info": info})
    finally:
        env.close()

play_env(seed=SEED)
